# Consistency Memory — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-5/consistency-memory.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-5/consistency-memory.ipynb)

Companion to: [Consistency Memory: Making Repeated AI Judgments Agree With Each Other](https://anacodicai.org/learn/ai-agents/6-5/consistency-memory)


## What you'll build

1. A judge that re-derives a score independently every time (with realistic
   sampling noise) — the normal, unmemoried behavior.
2. A **consistency memory** that stores the first judgment per category and
   reuses it for every later occurrence.
3. A side-by-side measurement of how much the two approaches actually disagree
   with themselves across 10 "identical" cases.

**No API key needed.** Sampling noise is simulated with `random`, seeded for
reproducibility.

### Setup

In [1]:
import random
import statistics
print("ready — standard library only")

ready — standard library only


## Step 1 — a judge with realistic sampling noise

Every call to `grade_without_memory` for the *same* error type returns a
slightly different number — this is what an LLM does by default: it samples,
it doesn't look things up.

In [2]:
def grade_without_memory(error_type, rng):
    base = {"off_by_one_index": 3.0, "missing_null_check": 2.0}[error_type]
    return round(base + rng.uniform(-1.2, 1.2), 1)

rng_demo = random.Random(1)
print("Five independent judgments of the SAME error type:")
print([grade_without_memory("off_by_one_index", rng_demo) for _ in range(5)])

Five independent judgments of the SAME error type:
[2.1, 3.8, 3.6, 2.4, 3.0]


## Step 2 — a memory store that reuses the first judgment

The first occurrence of a category is judged normally. Every later occurrence
of the same category looks the stored value up instead of re-deriving it —
and blends new occurrences in with a running average, so one noisy first draw
doesn't get frozen in forever.

In [3]:
class ConsistencyMemory:
    def __init__(self):
        self._store = {}

    def lookup(self, error_type):
        return self._store.get(error_type)

    def record(self, error_type, deduction, reasoning):
        if error_type in self._store:
            prev_deduction, _, count = self._store[error_type]
            blended = round((prev_deduction * count + deduction) / (count + 1), 1)
            self._store[error_type] = (blended, reasoning, count + 1)
        else:
            self._store[error_type] = (deduction, reasoning, 1)

def grade_with_memory(error_type, rng, memory):
    prior = memory.lookup(error_type)
    deduction = prior[0] if prior is not None else grade_without_memory(error_type, rng)
    memory.record(error_type, deduction, f"deduction for {error_type}")
    return deduction

print("ConsistencyMemory ready")

ConsistencyMemory ready


## Step 3 — ten "identical" submissions, both ways

**Try it:** change the seed in `random.Random(11)` below and re-run — the
*specific* numbers will change, but the shape of the result (no-memory spreads
out, with-memory locks to one value) will not.

In [4]:
rng_a = random.Random(11)
rng_b = random.Random(11)
memory = ConsistencyMemory()

submissions = ["off_by_one_index"] * 10
without_memory = [grade_without_memory(e, rng_a) for e in submissions]
with_memory = [grade_with_memory(e, rng_b, memory) for e in submissions]

print(f"{'submission':12}{'no memory':12}{'with memory':12}")
for i, (a, b) in enumerate(zip(without_memory, with_memory)):
    print(f"{i:<12}{a:<12}{b:<12}")

print(f"\nno-memory   stdev={statistics.pstdev(without_memory):.3f}  "
      f"(min={min(without_memory)}, max={max(without_memory)})")
print(f"with-memory stdev={statistics.pstdev(with_memory):.3f}  "
      f"(min={min(with_memory)}, max={max(with_memory)})")

submission  no memory   with memory 
0           2.9         2.9         
1           3.1         2.9         
2           4.0         2.9         
3           2.9         2.9         
4           3.0         2.9         
5           3.2         2.9         
6           2.2         2.9         
7           3.0         2.9         
8           3.3         2.9         
9           3.7         2.9         

no-memory   stdev=0.461  (min=2.2, max=4.0)
with-memory stdev=0.000  (min=2.9, max=2.9)


## Step 4 — prove the two properties that matter

1. With-memory judgments are identical from the second occurrence onward.
2. A *different* error category still gets its own independent judgment —
   memory doesn't bleed across categories.

In [5]:
assert with_memory[0] == with_memory[1] == with_memory[-1]
assert statistics.pstdev(with_memory) == 0.0
assert statistics.pstdev(without_memory) > 0.0
print("assert passed: with-memory is identical after occurrence 1; "
      "no-memory genuinely varies")

second_category = grade_with_memory("missing_null_check", rng_b, memory)
assert memory.lookup("off_by_one_index")[0] == with_memory[-1]
print(f"a different category ('missing_null_check') got its own judgment: "
      f"{second_category} — unrelated categories don't share memory")

assert passed: with-memory is identical after occurrence 1; no-memory genuinely varies
a different category ('missing_null_check') got its own judgment: 2.1 — unrelated categories don't share memory


## Try it yourself

- Widen the noise range in Step 1 (`rng.uniform(-1.2, 1.2)` → `(-3, 3)`) and
  re-run — the no-memory stdev in Step 3 should grow; the with-memory stdev
  stays exactly 0.0 regardless, because it's never re-sampling.
- Add a third, deliberately *mis-grouped* category that secretly contains two
  different real situations — this is the failure mode the article's §3.3
  and §8 warn about: memory forces them to share a judgment they shouldn't.